In [ ]:
# ============================================================
# PENELITIAN:
# Deteksi Serangan DDoS Minoritas menggunakan
# KNN Baseline vs KNN + ADASYN
#
# Dataset:
# data set pak fiki 1.csv
# ============================================================

# ============================================================
# 1. INSTALL LIBRARY
# ============================================================
# Jalankan jika library belum tersedia:
#
# pip install pandas numpy matplotlib seaborn scikit-learn imbalanced-learn
#
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter

# Scikit-learn
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Imbalanced-learn
from imblearn.over_sampling import ADASYN
from imblearn.pipeline import Pipeline as ImbPipeline


# ============================================================
# 2. KONFIGURASI
# ============================================================

DATASET = "data set pak fiki 1.csv"

# Random state untuk train-test split
RANDOM_STATE_SPLIT = 42

# Random state ADASYN
# Nilai 20 digunakan untuk mereproduksi hasil sebelumnya
RANDOM_STATE_ADASYN = 20

# KNN
K_NEIGHBORS = 7

# Folder hasil
OUTPUT_DIR = "hasil_knn_adasyn"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# 3. MEMBACA DATASET
# ============================================================

print("=" * 70)
print("MEMBACA DATASET")
print("=" * 70)

df = pd.read_csv(DATASET, sep=";")

print("\nUkuran dataset awal:")
print(df.shape)

print("\nNama kolom:")
print(df.columns.tolist())

print("\n5 data pertama:")
print(df.head())


# ============================================================
# 4. PEMERIKSAAN DATA
# ============================================================

print("\n" + "=" * 70)
print("PEMERIKSAAN DATA")
print("=" * 70)

print("\nInformasi data:")
print(df.info())

print("\nMissing value:")
print(df.isnull().sum())

print("\nJumlah duplikat:")
print(df.duplicated().sum())


# ============================================================
# 5. HANYA MENGGUNAKAN DATA YANG MEMILIKI LABEL
# ============================================================
#
# Jangan menganggap data tanpa label sebagai benign.
#
# Dataset memiliki:
# Benign = 1817
# DDoS   = 150
# Unlabeled = 8033
#
# Data unlabeled tidak digunakan dalam supervised learning.
# ============================================================

df_clean = df[df["label"].notna()].copy()

# Menghapus duplikat
df_clean = df_clean.drop_duplicates()

# Reset index
df_clean = df_clean.reset_index(drop=True)

# Pastikan label berupa integer
df_clean["label"] = df_clean["label"].astype(int)


print("\nJumlah data setelah cleaning:")
print(len(df_clean))

print("\nDistribusi label:")
print(df_clean["label"].value_counts())

print("\nDistribusi attack type:")
print(df_clean["attack_type"].value_counts())


# ============================================================
# 6. VALIDASI BAHWA ATTACK ADALAH DDOS
# ============================================================

print("\n" + "=" * 70)
print("DISTRIBUSI LABEL DAN ATTACK TYPE")
print("=" * 70)

print(
    pd.crosstab(
        df_clean["label"],
        df_clean["attack_type"]
    )
)


# ============================================================
# 7. FEATURE ENGINEERING
# ============================================================

print("\n" + "=" * 70)
print("FEATURE ENGINEERING")
print("=" * 70)

# ------------------------------------------------------------
# Timestamp
# ------------------------------------------------------------

df_clean["timestamp_parsed"] = pd.to_datetime(
    df_clean["timestamp"],
    format="%d/%m/%Y %H:%M",
    errors="coerce"
)

# Ekstraksi informasi waktu
df_clean["hour"] = df_clean["timestamp_parsed"].dt.hour
df_clean["minute"] = df_clean["timestamp_parsed"].dt.minute
df_clean["dayofweek"] = df_clean["timestamp_parsed"].dt.dayofweek
df_clean["day"] = df_clean["timestamp_parsed"].dt.day
df_clean["month"] = df_clean["timestamp_parsed"].dt.month


# ------------------------------------------------------------
# URL
# ------------------------------------------------------------

df_clean["url_len"] = (
    df_clean["url"]
    .fillna("")
    .astype(str)
    .str.len()
)

df_clean["url_has_query"] = (
    df_clean["url"]
    .fillna("")
    .astype(str)
    .str.contains(r"\?", regex=True)
    .astype(int)
)

df_clean["url_path_depth"] = (
    df_clean["url"]
    .fillna("")
    .astype(str)
    .str.count("/")
)

df_clean["url_has_http"] = (
    df_clean["url"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.startswith(("http://", "https://"))
    .astype(int)
)


# ------------------------------------------------------------
# User Agent
# ------------------------------------------------------------

df_clean["ua_len"] = (
    df_clean["user_agent"]
    .fillna("")
    .astype(str)
    .str.len()
)


# ============================================================
# 8. MENENTUKAN FITUR
# ============================================================

features = [

    # Time
    "hour",
    "minute",
    "dayofweek",
    "day",
    "month",

    # Network
    "src_port",
    "dst_port",
    "bytes_sent",
    "bytes_received",

    # URL
    "url_len",
    "url_has_query",
    "url_path_depth",
    "url_has_http",

    # User agent
    "ua_len",

    # Categorical
    "protocol",
    "user_agent",
    "is_internal_traffic"
]

X = df_clean[features].copy()

y = df_clean["label"].copy()


# ============================================================
# 9. CEK DISTRIBUSI DATA
# ============================================================

print("\n" + "=" * 70)
print("DISTRIBUSI DATA SEBELUM ADASYN")
print("=" * 70)

print(Counter(y))

print("\nPersentase kelas:")
print(
    y.value_counts(normalize=True) * 100
)


# ============================================================
# 10. TRAIN TEST SPLIT
# ============================================================
#
# 80% training
# 20% testing
#
# stratify=y memastikan proporsi kelas tetap terjaga.
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE_SPLIT,
    stratify=y
)

print("\n" + "=" * 70)
print("TRAIN TEST SPLIT")
print("=" * 70)

print("Training :", X_train.shape)
print("Testing  :", X_test.shape)

print("\nDistribusi training:")
print(Counter(y_train))

print("\nDistribusi testing:")
print(Counter(y_test))


# ============================================================
# 11. IDENTIFIKASI NUMERIC DAN CATEGORICAL FEATURES
# ============================================================

numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("\nNumeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)


# ============================================================
# 12. PREPROCESSING PIPELINE
# ============================================================

# Numeric:
# Missing value -> median
# StandardScaler

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


# Categorical:
# Missing value -> modus
# One Hot Encoding

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)


# Gabungkan preprocessing
preprocessor = ColumnTransformer(
    transformers=[

        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),

        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)


# ============================================================
# 13. KNN BASELINE
# ============================================================

knn_baseline = Pipeline(
    steps=[

        (
            "preprocessing",
            preprocessor
        ),

        (
            "knn",
            KNeighborsClassifier(
                n_neighbors=K_NEIGHBORS
            )
        )
    ]
)


# ============================================================
# 14. TRAIN KNN BASELINE
# ============================================================

print("\n" + "=" * 70)
print("TRAINING KNN BASELINE")
print("=" * 70)

knn_baseline.fit(
    X_train,
    y_train
)


# Prediksi
y_pred_knn = knn_baseline.predict(
    X_test
)


# ============================================================
# 15. EVALUASI KNN BASELINE
# ============================================================

accuracy_knn = accuracy_score(
    y_test,
    y_pred_knn
)

precision_knn = precision_score(
    y_test,
    y_pred_knn,
    zero_division=0
)

recall_knn = recall_score(
    y_test,
    y_pred_knn,
    zero_division=0
)

f1_knn = f1_score(
    y_test,
    y_pred_knn,
    zero_division=0
)

print("\nHASIL KNN BASELINE")
print("-" * 50)

print(f"Accuracy  : {accuracy_knn:.4f}")
print(f"Precision : {precision_knn:.4f}")
print(f"Recall    : {recall_knn:.4f}")
print(f"F1 Score  : {f1_knn:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_knn,
        target_names=["Benign", "DDoS"],
        zero_division=0
    )
)


# ============================================================
# 16. CONFUSION MATRIX KNN BASELINE
# ============================================================

cm_knn = confusion_matrix(
    y_test,
    y_pred_knn
)

print("\nConfusion Matrix KNN:")
print(cm_knn)


# ============================================================
# 17. KNN + ADASYN
# ============================================================
#
# ADASYN hanya bekerja pada data training.
#
# DATA TEST TIDAK BOLEH DI-ADASYN.
#
# Ini penting agar evaluasi tetap merepresentasikan
# data dunia nyata.
# ============================================================

knn_adasyn = ImbPipeline(
    steps=[

        (
            "preprocessing",
            preprocessor
        ),

        (
            "adasyn",
            ADASYN(
                random_state=RANDOM_STATE_ADASYN,
                n_neighbors=5
            )
        ),

        (
            "knn",
            KNeighborsClassifier(
                n_neighbors=K_NEIGHBORS
            )
        )
    ]
)


# ============================================================
# 18. TRAIN KNN + ADASYN
# ============================================================

print("\n" + "=" * 70)
print("TRAINING KNN + ADASYN")
print("=" * 70)

knn_adasyn.fit(
    X_train,
    y_train
)


# Prediksi
y_pred_adasyn = knn_adasyn.predict(
    X_test
)


# ============================================================
# 19. EVALUASI KNN + ADASYN
# ============================================================

accuracy_adasyn = accuracy_score(
    y_test,
    y_pred_adasyn
)

precision_adasyn = precision_score(
    y_test,
    y_pred_adasyn,
    zero_division=0
)

recall_adasyn = recall_score(
    y_test,
    y_pred_adasyn,
    zero_division=0
)

f1_adasyn = f1_score(
    y_test,
    y_pred_adasyn,
    zero_division=0
)


print("\nHASIL KNN + ADASYN")
print("-" * 50)

print(f"Accuracy  : {accuracy_adasyn:.4f}")
print(f"Precision : {precision_adasyn:.4f}")
print(f"Recall    : {recall_adasyn:.4f}")
print(f"F1 Score  : {f1_adasyn:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_adasyn,
        target_names=["Benign", "DDoS"],
        zero_division=0
    )
)


# ============================================================
# 20. CONFUSION MATRIX ADASYN
# ============================================================

cm_adasyn = confusion_matrix(
    y_test,
    y_pred_adasyn
)

print("\nConfusion Matrix KNN + ADASYN:")
print(cm_adasyn)


# ============================================================
# 21. TABEL PERBANDINGAN HASIL
# ============================================================

results = pd.DataFrame({

    "Model": [
        "KNN Baseline",
        "KNN + ADASYN"
    ],

    "Accuracy": [
        accuracy_knn,
        accuracy_adasyn
    ],

    "Precision": [
        precision_knn,
        precision_adasyn
    ],

    "Recall": [
        recall_knn,
        recall_adasyn
    ],

    "F1": [
        f1_knn,
        f1_adasyn
    ]
})


print("\n" + "=" * 70)
print("PERBANDINGAN MODEL")
print("=" * 70)

print(
    results.to_string(
        index=False
    )
)


# ============================================================
# 22. SIMPAN HASIL METRIK
# ============================================================

results.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "hasil_metrik.csv"
    ),
    index=False
)


# ============================================================
# 22B. PIE CHART DISTRIBUSI BENIGN VS ATTACK
# ============================================================

jumlah_benign = int((y == 0).sum())
jumlah_attack = int((y == 1).sum())

plt.figure(
    figsize=(7, 7)
)

plt.pie(
    [jumlah_benign, jumlah_attack],
    labels=[
        f"Benign ({jumlah_benign})",
        f"Attack ({jumlah_attack})"
    ],
    colors=[
        "#4C72B0",
        "#C44E52"
    ],
    autopct="%1.1f%%",
    startangle=0,
    explode=(0, 0.08),
    wedgeprops={
        "edgecolor": "white",
        "linewidth": 2
    },
    textprops={
        "fontsize": 12
    }
)

plt.title(
    "Distribusi Data Benign vs Attack",
    pad=20
)

plt.axis("equal")

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "pie_chart_benign_attack.svg"
    ),
    bbox_inches="tight",
    dpi=300
)

plt.show()


# ============================================================
# 23. VISUALISASI PERBANDINGAN METRIK
# ============================================================

metrics = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1"
]

x = np.arange(
    len(results["Model"])
)

width = 0.18

fig, ax = plt.subplots(
    figsize=(10, 6)
)

for i, metric in enumerate(metrics):

    ax.bar(
        x + (i - 1.5) * width,
        results[metric],
        width,
        label=metric
    )

ax.set_xlabel("Model")
ax.set_ylabel("Score")
ax.set_title(
    "Perbandingan KNN Baseline vs KNN + ADASYN"
)

ax.set_xticks(x)
ax.set_xticklabels(
    results["Model"]
)

ax.set_ylim(
    0,
    1.05
)

ax.legend()

ax.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "perbandingan_metrik.svg"
    ),
    bbox_inches="tight",
    dpi=300
)

plt.show()


# ============================================================
# 24. CONFUSION MATRIX KNN BASELINE
# ============================================================

plt.figure(
    figsize=(6, 5)
)

sns.heatmap(
    cm_knn,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=[
        "Benign",
        "DDoS"
    ],
    yticklabels=[
        "Benign",
        "DDoS"
    ]
)

plt.xlabel(
    "Predicted Label"
)

plt.ylabel(
    "Actual Label"
)

plt.title(
    "Confusion Matrix - KNN Baseline"
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "confusion_matrix_knn_baseline.svg"
    ),
    bbox_inches="tight",
    dpi=300
)

plt.show()


# ============================================================
# 25. CONFUSION MATRIX KNN + ADASYN
# ============================================================

plt.figure(
    figsize=(6, 5)
)

sns.heatmap(
    cm_adasyn,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=[
        "Benign",
        "DDoS"
    ],
    yticklabels=[
        "Benign",
        "DDoS"
    ]
)

plt.xlabel(
    "Predicted Label"
)

plt.ylabel(
    "Actual Label"
)

plt.title(
    "Confusion Matrix - KNN + ADASYN"
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "confusion_matrix_knn_adasyn.svg"
    ),
    bbox_inches="tight",
    dpi=300
)

plt.show()


# ============================================================
# 26. LEARNING CURVE KNN BASELINE
# ============================================================

train_sizes = np.linspace(
    0.2,
    1.0,
    5
)

train_sizes_knn, train_scores_knn, validation_scores_knn = (
    learning_curve(
        knn_baseline,
        X_train,
        y_train,

        train_sizes=train_sizes,

        cv=10,

        scoring="f1",

        n_jobs=-1
    )
)


# Rata-rata score
knn_train_mean = np.mean(
    train_scores_knn,
    axis=1
)

knn_validation_mean = np.mean(
    validation_scores_knn,
    axis=1
)


# ============================================================
# 27. LEARNING CURVE KNN + ADASYN
# ============================================================

train_sizes_ada, train_scores_ada, validation_scores_ada = (
    learning_curve(
        knn_adasyn,
        X_train,
        y_train,

        train_sizes=train_sizes,

        cv=5,

        scoring="f1",

        n_jobs=-1
    )
)


ada_train_mean = np.mean(
    train_scores_ada,
    axis=1
)

ada_validation_mean = np.mean(
    validation_scores_ada,
    axis=1
)


# ============================================================
# 28. VISUALISASI LEARNING CURVE
# ============================================================

plt.figure(
    figsize=(10, 6)
)

plt.plot(
    train_sizes_knn,
    knn_train_mean,
    marker="o",
    label="KNN Train F1"
)

plt.plot(
    train_sizes_knn,
    knn_validation_mean,
    marker="o",
    label="KNN Validation F1"
)

plt.plot(
    train_sizes_ada,
    ada_train_mean,
    marker="o",
    label="ADASYN Train F1"
)

plt.plot(
    train_sizes_ada,
    ada_validation_mean,
    marker="o",
    label="ADASYN Validation F1"
)

plt.xlabel(
    "Jumlah Sampel Training"
)

plt.ylabel(
    "F1 Score Kelas DDoS"
)

plt.title(
    "Learning Curve KNN vs KNN + ADASYN"
)

plt.grid(
    alpha=0.3
)

plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        "learning_curve_knn_adasyn.svg"
    ),
    bbox_inches="tight",
    dpi=300
)

plt.show()


# ============================================================
# 29. ANALISIS GAP TRAINING VS VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("ANALISIS LEARNING CURVE")
print("=" * 70)

print("\nKNN:")
print(
    "Train F1 terakhir     :",
    round(knn_train_mean[-1], 4)
)

print(
    "Validation F1 terakhir:",
    round(knn_validation_mean[-1], 4)
)

print(
    "Gap:",
    round(
        knn_train_mean[-1] -
        knn_validation_mean[-1],
        4
    )
)


print("\nKNN + ADASYN:")

print(
    "Train F1 terakhir     :",
    round(ada_train_mean[-1], 4)
)

print(
    "Validation F1 terakhir:",
    round(ada_validation_mean[-1], 4)
)

print(
    "Gap:",
    round(
        ada_train_mean[-1] -
        ada_validation_mean[-1],
        4
    )
)


# ============================================================
# 30. SIMPAN DATA CLEANING
# ============================================================

clean_output = df_clean.copy()

# Hapus kolom datetime sementara
if "timestamp_parsed" in clean_output.columns:
    clean_output = clean_output.drop(
        columns=["timestamp_parsed"]
    )

clean_output.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "dataset_cleaned_labeled.csv"
    ),
    index=False
)


# ============================================================
# 31. SIMPAN DATA FEATURE ENGINEERING
# ============================================================

processed_features = df_clean[
    features + ["label"]
].copy()

processed_features.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "dataset_processed_features.csv"
    ),
    index=False
)


# ============================================================
# 32. SELESAI
# ============================================================

print("\n" + "=" * 70)
print("EKSPERIMEN SELESAI")
print("=" * 70)

print("\nSemua hasil disimpan di folder:")
print(
    os.path.abspath(OUTPUT_DIR)
)

print("\nFile yang dihasilkan:")

for file in os.listdir(OUTPUT_DIR):
    print(
        "-",
        file
    )